## Inverse Kinematics (geometric, symbolic) - Lab 1 R-R-P robot
Given the tip position $(x, y, z)$, find $(q_1, q_2, q_3)$.

The robot has 3 joints, so only the tip **position** can be chosen (not its orientation).
Geometric IK depends on the robot's shape: the steps below are for this R-R-P robot.


Compare with `ik_algebraic.ipynb`: same answer, solved from the FK equations instead of a drawing.

In [1]:
import numpy as np
import sympy as sp
from IPython.display import display, Math
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3

def Rx(t): return sp.Matrix([[1, 0, 0, 0], [0, sp.cos(t), -sp.sin(t), 0], [0, sp.sin(t), sp.cos(t), 0], [0, 0, 0, 1]])
def Rz(t): return sp.Matrix([[sp.cos(t), -sp.sin(t), 0, 0], [sp.sin(t), sp.cos(t), 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
def Tx(d): return sp.Matrix([[1, 0, 0, d], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
def Tz(d): return sp.Matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, d], [0, 0, 0, 1]])

# Modified DH (Craig): T^{i-1}_i = Rx(alpha_{i-1}) Tx(a_{i-1}) Rz(theta_i) Tz(d_i)
def mdh(a, alpha, d, theta):
    return Rx(alpha) * Tx(a) * Rz(theta) * Tz(d)

# display only: sin(q1) -> s_1, cos(q1 + q2) -> c_12
def short(e):
    def sub(f):
        terms = sp.Add.make_args(f.args[0])
        if not all(t.is_Symbol and t.name[0] == 'q' for t in terms):
            return f
        idx = ''.join(sorted((t.name[1:] for t in terms), key=int))
        return sp.Symbol(('s' if f.func == sp.sin else 'c') + '_{' + idx + '}')
    return e.replace(lambda t: t.func in (sp.sin, sp.cos), sub)

def show(lhs, rhs):
    tex = sp.latex(short(rhs)).replace(r'\operatorname{atan}_{2}', r'\operatorname{atan2}')
    display(Math(lhs + ' = ' + tex))

### 0 - Forward kinematics (position only)
Same MDH table as `fk.ipynb`. IK has to undo this $p_e$.

In [2]:
q1, q2, q3 = sp.symbols('q1:4', real=True)
l1, l2, l3, l4 = sp.symbols('l1:5', positive=True)
x, y, z = sp.symbols('x y z', real=True)   # target tip position

# MDH table -> (a_{i-1}, alpha_{i-1}, d_i, theta_i)
mdh_table = [
    (0, 0,        l1,      q1),
    (0, sp.pi/2, -l2,      q2 + sp.pi/2),
    (0, sp.pi/2,  q3 + l3, 0),
]
T_tool = Tz(l4)

T0e = sp.eye(4)
for row in mdh_table:
    T0e = T0e * mdh(*row)
T0e = sp.simplify(T0e * T_tool)
p = T0e[0:3, 3]

show(r'p_e', p)

<IPython.core.display.Math object>

### 1 - Total reach $L$
Joint 3 slides along the arm, and $l_3$, $q_3$, $l_4$ all lie on that same line.
Call the total length from joint 2 to the tip $L = l_3 + q_3 + l_4$.

In [3]:
L = sp.Symbol('L', positive=True)
p_L = sp.simplify(p.subs(q3, L - l3 - l4))

show(r'L', l3 + q3 + l4)
show(r'p_e', p_L)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 2 - Top view (x-y plane): horizontal reach $\rho$
Seen from above, the arm reaches out $\rho = L c_2$ and is shifted sideways by $l_2$ (always perpendicular).
These two legs form a right triangle with $\sqrt{x^2 + y^2}$, so $\rho = \pm\sqrt{x^2 + y^2 - l_2^2}$.
The $\pm$ gives **2 branches** (arm reaching forward, or flipped over the top to reach backward).
Reachable only if $x^2 + y^2 \ge l_2^2$.

In [4]:
rho = sp.Symbol('rho', real=True)

show(r'x^2 + y^2', sp.simplify(sp.expand(p_L[0]**2 + p_L[1]**2)))
rho_sol = sp.sqrt(x**2 + y**2 - l2**2)
display(Math(r'\rho = L c_2 = \pm' + sp.latex(rho_sol)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 3 - Top view: $q_1$
$(x, y)$ is the vector $(\rho, l_2)$ rotated by $q_1$.
So the angle of the tip, $\operatorname{atan2}(y, x)$, is $q_1$ plus the angle of $(\rho, l_2)$.

In [5]:
show(r'\begin{bmatrix} x \\ y \end{bmatrix}', sp.Matrix([[sp.cos(q1), -sp.sin(q1)], [sp.sin(q1), sp.cos(q1)]]) * sp.Matrix([rho, l2]))

q1_sol = sp.atan2(y, x) - sp.atan2(l2, rho)
show(r'q_1', q1_sol)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 4 - Side view (arm plane): $q_2$ and $q_3$
In the vertical plane of the arm, the tip is $\rho$ out and $z - l_1$ up from joint 2,
at distance $L$ and elevation angle $q_2$.

In [6]:
show(r'\begin{bmatrix} \rho \\ z - l_1 \end{bmatrix}', sp.Matrix([L * sp.cos(q2), L * sp.sin(q2)]))

q2_sol = sp.atan2(z - l1, rho)
L_sol = sp.sqrt(rho**2 + (z - l1)**2)
q3_sol = L_sol - l3 - l4

show(r'q_2', q2_sol)
show(r'L', L_sol)
show(r'q_3 = L - l_3 - l_4', q3_sol)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5 - IK solution in terms of $(x, y, z)$

In [7]:
solutions = []
for k, sign in enumerate((1, -1), start=1):
    sol = sp.Matrix([q1_sol, q2_sol, q3_sol]).subs(rho, sign * rho_sol)
    solutions.append(sol)
    sgn = '+' if sign > 0 else '-'
    display(Math(rf'\text{{Branch {k}}}\ (\rho = {sgn}\sqrt{{x^2 + y^2 - l_2^2}}):'))
    show(r'\begin{bmatrix} q_1 \\ q_2 \\ q_3 \end{bmatrix}', sol)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 6 - Check with numbers
Pick joint values, get the tip with FK, solve IK for that tip, then FK the IK answer again: the error should be ~0.

In [8]:
# edit here: link lengths (m)
vals = {l1: 0.64, l2: 0.2, l3: 0.22, l4: 0.095}

robot_mdh = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0,     d=vals[l1]),
    RevoluteMDH(a=0.0, alpha=np.pi/2, d=-vals[l2], offset=np.pi/2),
    PrismaticMDH(a=0.0, alpha=np.pi/2, offset=vals[l3]),
], name='my_mdh')
robot_mdh.tool = SE3.Tz(vals[l4])

ik = [sp.lambdify((x, y, z), sol.subs(vals), 'numpy') for sol in solutions]

# edit here: joint values to test
for q_true in ([np.pi/6, np.pi/6, 0.18], [np.deg2rad(-25), np.deg2rad(-25), 0.135]):
    P = robot_mdh.fkine(q_true).t
    print('q =', np.round(q_true, 4), '-> tip P =', np.round(P, 4))
    for k, f in enumerate(ik, start=1):
        q_ik = np.array(f(*P), dtype=float).flatten()
        err = np.linalg.norm(robot_mdh.fkine(q_ik).t - P)
        print(f'  branch {k}: q = {np.round(q_ik, 4)} | FK error = {err:.1e} m')

q = [0.5236 0.5236 0.18  ] -> tip P = [0.2713 0.3875 0.8875]
  branch 1: q = [0.5236 0.5236 0.18  ] | FK error = 5.6e-17 m
  branch 2: q = [-1.7449  2.618   0.18  ] | FK error = 1.4e-16 m
q = [-0.4363 -0.4363  0.135 ] -> tip P = [0.4542 0.0089 0.4498]
  branch 1: q = [-0.4363 -0.4363  0.135 ] | FK error = 3.5e-17 m
  branch 2: q = [-2.6661 -2.7053  0.135 ] | FK error = 1.1e-16 m


### Numeric example
Put a target $(x, y, z)$ through every step above (both branches), then FK the answer to check.

In [9]:
# helpers: plug numbers into a symbolic result and print it rounded
def val(e, subs):
    return float(sp.sympify(e).subs(subs).evalf())
def mat(e, subs, d=4):
    return sp.Matrix(e).subs(subs).evalf().applyfunc(lambda v: sp.Float(round(float(v), d)) if abs(float(v)) > 1e-12 else 0)
def showv(lhs, v, unit=''):
    display(Math(rf'{lhs} = {v:.4f}\ \text{{{unit}}}'))
def showa(lhs, v):
    display(Math(rf'{lhs} = {v:.4f}\ \text{{rad}} = {np.rad2deg(v):.2f}^\circ'))
def showm(lhs, M):
    display(Math(lhs + ' = ' + sp.latex(M)))

# edit here: target tip position (m). Here it comes from FK of a known pose, or type numbers directly
target = robot_mdh.fkine([np.pi/6, np.pi/6, 0.18]).t        # e.g. target = [0.27, 0.39, 0.89]

nums = {**vals, x: target[0], y: target[1], z: target[2]}
showm(r'\begin{bmatrix} x \\ y \\ z \end{bmatrix}', mat(sp.Matrix(target), {}))
showv(r'\sqrt{x^2 + y^2 - l_2^2}', val(rho_sol, nums), 'm')

for k, sign in enumerate((1, -1), start=1):
    display(Math(rf'\textbf{{Branch {k}}}:'))
    n_k = {**nums, rho: sign * val(rho_sol, nums)}
    showv(r'\rho', n_k[rho], 'm')
    q1v = val(q1_sol, n_k)
    q2v = val(q2_sol, n_k)
    Lv = val(L_sol, n_k)
    q3v = Lv - vals[l3] - vals[l4]
    showa(r'q_1 = \operatorname{atan2}(y, x) - \operatorname{atan2}(l_2, \rho)', q1v)
    showa(r'q_2 = \operatorname{atan2}(z - l_1, \rho)', q2v)
    showv(r'L = \sqrt{\rho^2 + (z - l_1)^2}', Lv, 'm')
    showv(r'q_3 = L - l_3 - l_4', q3v, 'm')
    err = np.linalg.norm(robot_mdh.fkine([q1v, q2v, q3v]).t - target)
    print(f'  FK check: |FK(q) - target| = {err:.1e} m')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

... (6 outputs left out)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

  FK check: |FK(q) - target| = 1.4e-16 m


### Same with Robotics Toolbox (numerical IK)
`ikine_LM` searches numerically instead of using a formula. Only the position can be chosen (3 joints), so `mask` keeps $x, y, z$ only.
The start guess `q0` decides which branch it finds.
The cell declares its own MDH robot first, so it can be copied and run on its own (lengths in m: l1 = 0.64, l2 = 0.2, l3 = 0.22, l4 = 0.095).


In [10]:
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3

# 1) declare the robot with MDH (a = a_{i-1}, alpha = alpha_{i-1}, d = d_i, offset added to the joint variable)
robot = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0,     d=0.64,                    qlim=[-np.pi, np.pi]),   # joint 1 (R)
    RevoluteMDH(a=0.0, alpha=np.pi/2, d=-0.2,  offset=np.pi/2,   qlim=[-np.pi, np.pi]),   # joint 2 (R)
    PrismaticMDH(a=0.0, alpha=np.pi/2, offset=0.22,              qlim=[0.0, 0.3]),        # joint 3 (P), m
], name='my_mdh')
robot.tool = SE3.Tz(0.095)                  # l4: last joint -> tip
print(robot)

# 2) use it
Tep = SE3.Trans(target)                     # wanted tip pose (only the position is used)
mask = [1, 1, 1, 0, 0, 0]                   # match x, y, z; ignore orientation

# edit here: start guesses (one near each branch)
for q0 in ([0.0, 0.0, 0.1], [-1.5, 2.5, 0.1]):
    sol = robot.ikine_LM(Tep, q0=q0, mask=mask, tol=1e-12)   # smaller tol = more exact
    q_sol = sol.q
    err = np.linalg.norm(robot.fkine(q_sol).t - target)
    print(f'q0 = {q0} -> success = {sol.success}')
    print(f'  q = [{np.rad2deg(q_sol[0]):.2f} deg, {np.rad2deg(q_sol[1]):.2f} deg, {q_sol[2]:.4f} m] | FK error = {err:.1e} m')

DHRobot: my_mdh, 3 joints (RRP), modified DH parameters
┌──────┬───────┬───────────┬────────────┬─────────┬────────┐
│ aⱼ₋₁ │ ⍺ⱼ₋₁  │    θⱼ     │     dⱼ     │   q⁻    │   q⁺   │
├──────┼───────┼───────────┼────────────┼─────────┼────────┤
│  0.0 │  0.0° │        q1 │       0.64 │ -180.0° │ 180.0° │
│  0.0 │ 90.0° │  q2 + 90° │       -0.2 │ -180.0° │ 180.0° │
│  0.0 │ 90.0° │      0.0° │  q3 + 0.22 │       0 │    0.3 │
└──────┴───────┴───────────┴────────────┴─────────┴────────┘

┌──────┬───────────────────────────────────────┐
│ tool │ t = 0, 0, 0.095; rpy/xyz = 0°, 0°, 0° │
└──────┴───────────────────────────────────────┘

q0 = [0.0, 0.0, 0.1] -> success = True
  q = [30.00 deg, 30.00 deg, 0.1800 m] | FK error = 4.8e-07 m
q0 = [-1.5, 2.5, 0.1] -> success = True
  q = [-99.98 deg, 150.00 deg, 0.1800 m] | FK error = 3.0e-07 m
